# ETF Pattern Matching with DTW — Rust/PyO3 Interactive Demo

> **Project**: [etf-pattern-match-pyo3](https://github.com/redamancy231-create/etf-pattern-match-pyo3)  
> **License**: MIT  
> **Backend**: Rust 1.97+ · PyO3 · rayon  
> **Validation**: 31 golden fixtures · 14 Rust tests · 62 Python tests

---

## What This Notebook Covers

1. **The problem**: finding historical price patterns that resemble the current market
2. **The algorithm**: two-stage matching — cosine pre-filter → DTW refinement
3. **The features**: 15-dimensional morphological signature for each match
4. **The backend**: local Rust/PyO3 timing plus the preregistered NRR-2026-023 summary
5. **Practical use**: scanning multiple time points for illustrative signals

---

## 本 Notebook 涵盖内容

1. **问题**：寻找与当前市场形态相似的历史价格模式
2. **算法**：两阶段形态匹配——余弦预筛选 → DTW 精排
3. **特征**：每个匹配的 15 维形态签名
4. **后端**：本地 Rust/PyO3 计时与 NRR-2026-023 预登记结果摘要
5. **实战**：扫描多个时点生成仅供演示的信号


## 0. Setup & Imports

In [ ]:
import time
from datetime import datetime, timedelta
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from etf_pattern_match_pyo3 import (
    FEATURE_KEYS,
    cosine_similarity,
    dtw_distance,
    pattern_match_batch,
    pattern_match_single,
    standardize_returns,
)


def generate_query_candidates(
    prices: np.ndarray,
    t_idx: int,
    L_query: int = 20,
    T_back: int = 750,
    match_step: int = 1,
    M_forward: int = 5,
):
    """生成演示用历史窗口；距离与完整匹配均由 Rust 后端计算。"""
    query_start = t_idx - L_query + 1
    if query_start < 0:
        raise ValueError("insufficient query history")
    earliest_end = max(L_query - 1, t_idx - T_back)
    latest_end = query_start - M_forward - 1
    candidate_ends = np.arange(earliest_end, latest_end + 1, match_step, dtype=np.int64)
    candidates = np.asarray(
        [prices[end - L_query + 1 : end + 1] for end in candidate_ends],
        dtype=np.float64,
    )
    return prices[query_start : t_idx + 1], candidates, candidate_ends


def dtw_distance_batch(
    query: np.ndarray,
    candidates: np.ndarray,
    window: int = 5,
    top_k: int = 10,
):
    """Notebook 辅助函数：逐候选调用 Rust `dtw_distance` 后取 Top-K。"""
    distances = np.asarray(
        [dtw_distance(query, np.ascontiguousarray(candidate), window) for candidate in candidates],
        dtype=np.float64,
    )
    order = np.argsort(distances, kind="stable")[:top_k]
    return order, distances[order]


plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 100,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("✅ etf_pattern_match_pyo3 Rust/PyO3 backend loaded")
print(f"NumPy version: {np.__version__}")
print(f"FEATURE_KEYS: {len(FEATURE_KEYS)}")


## 1. Generate Synthetic ETF Price Data

We'll create a realistic price series with trends, mean-reversion, and volatility clustering — mimicking actual ETF behavior. This lets us test the algorithm without needing a market data API.

> **Data generation note**: Prices are generated from Gaussian simple returns with rolling volatility feedback (not log-normal returns or true GARCH). The dates use consecutive calendar days for simplicity — not actual trading-day calendars.

In [ ]:
def generate_realistic_etf_prices(
    n_days: int = 800,
    start_price: float = 100.0,
    mu: float = 0.0003,          # daily drift (~8% annual)
    sigma: float = 0.012,         # daily volatility (~19% annual)
    regime_change_prob: float = 0.005,  # probability of regime switch per day
    seed: int = 42,
) -> np.ndarray:
    """
    Generate realistic ETF-like price series with:
    - Gaussian simple returns (not log-normal)
    - Rolling-window volatility feedback (simplified, not true GARCH)
    - Occasional trend reversals (regime changes)
    """
    rng = np.random.default_rng(seed)

    returns = np.zeros(n_days)
    current_mu = mu
    vol_window = 20

    for i in range(n_days):
        # Regime switch — occasional trend change
        if rng.random() < regime_change_prob:
            current_mu = rng.normal(0, 0.001)  # new drift

        # Volatility clustering: recent vol affects current vol
        if i >= vol_window:
            recent_vol = np.std(returns[i - vol_window:i])
            effective_sigma = 0.7 * sigma + 0.3 * recent_vol
        else:
            effective_sigma = sigma

        returns[i] = rng.normal(current_mu, effective_sigma)

    # Cumulative product to get prices
    prices = start_price * np.cumprod(1.0 + returns)
    return np.asarray(prices, dtype=np.float64)


# Generate data
prices = generate_realistic_etf_prices(n_days=800, seed=42)
dates = [datetime(2023, 1, 1) + timedelta(days=i) for i in range(len(prices))]

print(f"Generated {len(prices)} synthetic observations with calendar-day spacing")
print(f"  Start:  ${prices[0]:.2f}")
print(f"  End:    ${prices[-1]:.2f}")
print(f"  Return: {prices[-1]/prices[0] - 1:.1%}")
print(f"  Vol:    {np.std(np.diff(np.log(prices))):.2%} daily")

In [ ]:
# Visualize the full price series
fig, axes = plt.subplots(2, 1, figsize=(15, 7), height_ratios=[3, 1])

# Price chart
axes[0].plot(dates, prices, linewidth=0.8, color="#1a3a5c")
axes[0].set_title("Synthetic ETF Price Series (800 observations, calendar-day spacing)")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True, alpha=0.3)

# Daily returns
daily_rets = np.diff(np.log(prices))
axes[1].bar(dates[1:], daily_rets, width=1.0, color=["#c0392b" if r < 0 else "#27ae60" for r in daily_rets], alpha=0.6)
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].set_ylabel("Daily Return")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_price_series.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('01_price_series.png'))

## 2. The Core Algorithm: Two-Stage Pattern Matching

### How It Works

```
    Query Window                Historical Search Space
    [ T-19 .. T ]               [ T-750 .. T-20 ]
         │                              │
         ▼                              ▼
  ┌──────────────┐           ┌──────────────────────┐
  │  Stage 1:    │──────────▶│  Cosine Pre-filter    │
  │  Standardize │           │  ~750 candidates O(L) │
  │  Returns     │           │  → Keep top-50        │
  └──────────────┘           └──────────┬───────────┘
                                        │
                                        ▼
                               ┌──────────────────────┐
                               │  Stage 2:            │
                               │  DTW Refinement      │
                               │  50 candidates O(L²) │
                               │  → Top-K matches     │
                               └──────────┬───────────┘
                                        │
                                        ▼
                               ┌──────────────────────┐
                               │  15-Dimensional      │
                               │  Feature Extraction  │
                               └──────────────────────┘
```

**Why two stages?** Computing DTW for all 750 candidates is O(750 × L²). Cosine similarity is O(L). By filtering to top-50 first, we get ~93% compute savings while retaining the best candidates.

**Look-ahead bias prevention**: All matching windows end at `search_end = T_idx - L_query`, ensuring we never use future data. The causal boundary is strictly enforced: `fut_end = hist_end + M_forward < T_idx`.

### 2.1 Step 1: Define Query Window & Standardize Returns

In [ ]:
# Pick an analysis point
T_idx = 500  # day 500 of 800
L_query = 20  # 20-day query window

# Standardize: (log returns - mean) / std
query_prices = prices[T_idx - L_query + 1 : T_idx + 1]
query_rets = standardize_returns(query_prices)

print(f"Query window:  days [{T_idx - L_query + 1}, {T_idx}]")
print(f"Query prices:  {L_query} data points")
print(f"Query returns: {len(query_rets)} standardized log-returns")
print(f"  Mean: {np.mean(query_rets):.6f}  (≈ 0 after standardization)")
print(f"  Std:  {np.std(query_rets):.6f}   (≈ 1 after standardization)")

In [ ]:
# Visualize: Query window in historical context
fig, axes = plt.subplots(2, 1, figsize=(15, 7))

# Top: Full price series with query window highlighted
query_start = T_idx - L_query + 1
axes[0].plot(dates, prices, linewidth=0.5, color="gray", alpha=0.5, label="Full series")
axes[0].plot(dates[query_start:T_idx+1], prices[query_start:T_idx+1], 
             linewidth=2.5, color="#e74c3c", label=f"Query window (L={L_query})")
axes[0].axvline(x=dates[T_idx], color="#e74c3c", linestyle="--", alpha=0.5, label=f"T_idx={T_idx}")
axes[0].set_title("Query Window in Historical Context")
axes[0].set_ylabel("Price ($)")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# Bottom: Standardized returns — this is what we match on
axes[1].bar(range(len(query_rets)), query_rets, color="#e74c3c", alpha=0.7)
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].set_title("Standardized Log-Returns of Query Window (shape to match)")
axes[1].set_xlabel("Day offset within window")
axes[1].set_ylabel("Standardized Return")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('02_query_window.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('02_query_window.png'))

### 2.2 Step 2: Cosine Pre-Filter — Scan All Historical Windows

In [ ]:
# Generate candidate windows from history
q_prices, candidates_prices, candidate_ends = generate_query_candidates(
    prices, T_idx, L_query=L_query, T_back=750, match_step=1
)

# Standardize all candidate returns and compute cosine similarity + fast shape distances
query_r = standardize_returns(q_prices)
cos_sims = np.empty(len(candidates_prices))
fast_shape_dists = np.empty(len(candidates_prices))  # RMSD for all candidates

for i in range(len(candidates_prices)):
    cand_r = standardize_returns(candidates_prices[i])
    cos_sims[i] = cosine_similarity(cand_r, query_r)
    # Fast shape distance (RMSD) — used by real API for sigma calibration
    fast_shape_dists[i] = np.sqrt(np.mean((cand_r - query_r) ** 2))

# Keep only cos > 0 candidates + top-N by cosine
positive_mask = cos_sims > 0
cos_positive = cos_sims[positive_mask]
cos_prefilter_top = 50
top_n = min(cos_prefilter_top, len(cos_positive))
top_cos_idx = np.argsort(cos_positive)[::-1][:top_n]

print(f"Total candidates scanned: {len(candidates_prices)}")
print(f"Cosine > 0:              {np.sum(positive_mask)}")
print(f"Top-{top_n} by cosine retained for DTW refinement")
print(f"\nCosine similarity range: [{cos_sims.min():.3f}, {cos_sims.max():.3f}]")

In [ ]:
# Visualize cosine pre-filter results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Cosine similarity histogram
axes[0].hist(cos_sims, bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(x=0, color="#e74c3c", linestyle="--", linewidth=1.5, label="cos = 0")
axes[0].set_title(f"Cosine Similarity Distribution ({len(cos_sims)} candidates)")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Top-3 best cosine matches vs query (raw prices)
colors = ["#e74c3c", "#2ecc71", "#3498db"]
axes[1].plot(query_prices, linewidth=3, color="black", label="Query", zorder=10)
for rank, (ci, color) in enumerate(zip(top_cos_idx[:3], colors)):
    # Find original index in candidates_prices
    orig_idx = np.where(positive_mask)[0][ci]
    axes[1].plot(candidates_prices[orig_idx], linewidth=1.5, color=color, alpha=0.7,
                 label=f"Match #{rank+1} (cos={cos_positive[ci]:.3f})")
axes[1].set_title("Top-3 Cosine Matches vs Query (Raw Prices)")
axes[1].set_xlabel("Day offset")
axes[1].set_ylabel("Price ($)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_cosine_prefilter.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('03_cosine_prefilter.png'))

### 2.3 Step 3: DTW Refinement — Warp-Aware Matching

Cosine similarity measures 
**point-by-point alignment** — day 1 vs day 1, day 2 vs day 2.  
DTW allows **non-linear warping** — a 3-day rally in history can match a 2-day rally today.

This is critical for financial patterns: markets speed up and slow down, but the *shape* repeats.

In [ ]:
# DTW compute on top cosine candidates
dtw_window = 5  # Sakoe-Chiba band
dtw_dists = []

for ci in top_cos_idx:
    orig_idx = np.where(positive_mask)[0][ci]
    cand_r = standardize_returns(candidates_prices[orig_idx])
    d = dtw_distance(cand_r, query_r, window=dtw_window)
    dtw_dists.append(d)

dtw_dists = np.array(dtw_dists)
cos_top = cos_positive[top_cos_idx]

# ── Scoring matches the real pattern_match_single implementation ──
# sigma_fast: calibrated from ALL candidates' fast_shape_dists (RMSD),
# scaled by 1/(2*sqrt(L-1)) to match DTW's normalization convention
sigma_fast = (
    np.std(fast_shape_dists) / (2.0 * np.sqrt(L_query - 1))
    if len(fast_shape_dists) > 1
    else 1.0
)
sigma_fast = max(sigma_fast, 1e-12)

# RBF kernel with sigma_fast → min-max normalize (global cos bounds) → 0.5*DTW + 0.5*cosine
sim_dtw = np.exp(-dtw_dists / sigma_fast)
norm_dtw = (sim_dtw - sim_dtw.min()) / (sim_dtw.max() - sim_dtw.min() + 1e-12)
norm_cos = (cos_top - cos_positive.min()) / (cos_positive.max() - cos_positive.min() + 1e-12)
combined = 0.5 * norm_dtw + 0.5 * norm_cos

print(f"DTW refinement on top-{top_n} candidates:")
print(f"  sigma_fast: {sigma_fast:.6f}  (calibrated from {len(fast_shape_dists)} candidates)")
print(f"  DTW distances:  [{dtw_dists.min():.4f}, {dtw_dists.max():.4f}]")
print(f"  Combined score: [{combined.min():.3f}, {combined.max():.3f}]")

In [ ]:
# ⚠️ Educational note: The DTW matrix below is computed WITHOUT the Sakoe-Chiba
# band constraint (window=∞) to show the full warping path for visualization.
# The actual distance value reported in the title uses the constrained DTW
# (window=5) — as does the real pattern_match_single API.
#
# Visualize DTW warping path for best match
best_idx = np.argmax(combined)
best_cand_idx = np.where(positive_mask)[0][top_cos_idx[best_idx]]
best_cand_r = standardize_returns(candidates_prices[best_cand_idx])

# Compute FULL DTW matrix for visualization (unconstrained — clearer warping picture)
n, m = len(query_r), len(best_cand_r)
dtw_mat = np.full((n + 1, m + 1), np.inf)
dtw_mat[0, 0] = 0.0
for i in range(1, n + 1):
    for j in range(1, m + 1):
        cost = (query_r[i-1] - best_cand_r[j-1]) ** 2
        dtw_mat[i, j] = cost + min(dtw_mat[i-1, j], dtw_mat[i, j-1], dtw_mat[i-1, j-1])

# Backtrack for warping path
path = [(n, m)]
i, j = n, m
while i > 0 or j > 0:
    if i == 0:
        j -= 1
    elif j == 0:
        i -= 1
    else:
        steps = [(i-1, j-1, dtw_mat[i-1, j-1]), (i-1, j, dtw_mat[i-1, j]), (i, j-1, dtw_mat[i, j-1])]
        best_step = min(steps, key=lambda x: x[2])
        i, j = best_step[0], best_step[1]
    path.append((i, j))
path = np.array(path)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: DTW matrix + warping path
im = axes[0].imshow(dtw_mat[1:, 1:], origin="upper", cmap="YlOrRd", aspect="auto")
axes[0].plot(path[:, 1] - 1, path[:, 0] - 1, color="blue", linewidth=1.5, alpha=0.8)
axes[0].plot([0, m-1], [0, n-1], color="white", linestyle="--", linewidth=0.8, alpha=0.5, label="Diagonal (no warping)")
axes[0].set_title(f"DTW Cost Matrix + Warping Path (full, unconstrained)\nBest match, constrained dtw_dist={dtw_dists[best_idx]:.4f}")
axes[0].set_xlabel("Candidate day")
axes[0].set_ylabel("Query day")
axes[0].legend(fontsize=8)
plt.colorbar(im, ax=axes[0], label="Cost")

# Right: Query vs Best Match returns (with warping connections)
x_q = np.arange(len(query_r))
x_c = np.arange(len(best_cand_r))
axes[1].plot(x_q, query_r, "o-", linewidth=2, markersize=5, color="black", label="Query")
axes[1].plot(x_c, best_cand_r, "s-", linewidth=2, markersize=5, color="#3498db", label="Best match")
# Draw a few warping connections
for step_idx in np.linspace(0, len(path)-1, 15, dtype=int):
    qi, ci = path[step_idx]
    if qi > 0 and ci > 0:
        axes[1].plot([ci-1, qi-1], [best_cand_r[ci-1], query_r[qi-1]],
                     color="gray", alpha=0.2, linewidth=0.5, zorder=0)
axes[1].set_title("Query vs Best DTW Match (Standardized Returns)")
axes[1].set_xlabel("Day offset")
axes[1].set_ylabel("Standardized Return")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_dtw_warping.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('04_dtw_warping.png'))

## 3. One-Call API: `pattern_match_single()`

All the steps above are packaged into a single call that returns the **15-dimensional morphological feature vector**.

In [ ]:
# The full algorithm in ONE call
features = pattern_match_single(prices, T_idx=500)

if features:
    # Group features by category
    similarity_features = ["top1_sim", "top5_avg_sim", "sim_decay", "sim_variance", "match_distance_ratio"]
    future_ret_features = ["avg_future_ret", "weighted_future_ret", "median_future_ret",
                           "ret_sign_consistency", "best_match_ret", "max_dd_in_matches"]
    quality_features = ["match_time_span", "match_time_span_ratio", "match_cluster_ratio", "n_matches_above_thresh"]
    
    print("=" * 60)
    print("15-DIMENSIONAL MORPHOLOGICAL FEATURE VECTOR")
    print("=" * 60)
    
    print("\n📊 F1-F5: Similarity Features")
    print("-" * 40)
    for key in similarity_features:
        print(f"  {key:28s} = {features[key]:.6f}")
    
    print("\n📈 F6-F11: Forward Return Features")
    print("-" * 40)
    for key in future_ret_features:
        print(f"  {key:28s} = {features[key]:.6f}")
    
    print("\n🔍 F12-F15: Match Quality Features")
    print("-" * 40)
    for key in quality_features:
        print(f"  {key:28s} = {features[key]:.6f}")
else:
    print("❌ Insufficient data at T_idx=500")

### Feature Interpretation Guide

| # | Feature | Range | What It Tells You |
|:--|:--------|:------|:-------------------|
| **F1** | `top1_sim` | [0, 1] | Best match quality. >0.8 = strong pattern |
| **F2** | `top5_avg_sim` | [0, 1] | Average of top-5. Close to F1 = many good matches |
| **F3** | `sim_decay` | [0, 1] | F1 − F2. Large = one standout match dominates |
| **F4** | `sim_variance` | [0, ~0.25] | Score dispersion. High = uncertain match quality |
| **F5** | `match_distance_ratio` | [0, 1] | Relative score decay (F3 / F1). >0.5 = the top match is isolated from the rest |
| **F6** | `avg_future_ret` | (−1, +∞) | Simple average of forward returns after historical matches |
| **F7** | `weighted_future_ret` | (−1, +∞) | Score-weighted forward return (better matches weighted higher) |
| **F8** | `median_future_ret` | (−1, +∞) | Robust center of forward return distribution |
| **F9** | `ret_sign_consistency` | [0, 1] | Proportion of matches with positive forward returns. 0 = all bearish (perfectly consistent bearish), 1 = all bullish. Use `max(p, 1-p)` for directional consistency. |
| **F10** | `best_match_ret` | (−1, +∞) | Forward return after the single highest-scoring match |
| **F11** | `max_dd_in_matches` | [0, +∞) | Worst loss among match forward returns (= `max(0, −min(rets))`). Note: this is the worst single-period forward loss, NOT a path-based maximum drawdown. |
| **F12** | `match_time_span` | [0, T_back−L] | Index span between oldest and newest match (bar count, not calendar days) |
| **F13** | `match_time_span_ratio` | [0, ~0.97] | F12 / T_back. High = matches drawn from across the full lookback |
| **F14** | `match_cluster_ratio` | [1/K_actual, 1] | Max fraction of matches within a 60-bar window. >0.5 = matches are concentrated in one period (potential overfitting) |
| **F15** | `n_matches_above_thresh` | [0, K_actual] | Count of matches with combined score > 0.8. <2 = weak pattern confidence |

> **Range notes**: F6–F10 use simple returns `p[t+n]/p[t] − 1`, whose theoretical range is (−1, +∞) for positive prices. K_actual is the effective K after NaN-filtering and may be less than configured K.

### 3.1 Visualize Feature Evolution Over Time

Scan multiple time points to see how the 15-dimensional signature changes as the market evolves.

In [ ]:
# Scan multiple time points
scan_points = list(range(200, 750, 10))
all_features: List[Optional[Dict[str, float]]] = []

for T in scan_points:
    all_features.append(pattern_match_single(prices, T))

# Extract key features over time
valid_indices = [i for i, f in enumerate(all_features) if f is not None]
valid_dates = [dates[scan_points[i]] for i in valid_indices]

top1_sim_series = [all_features[i]["top1_sim"] for i in valid_indices]
avg_future_ret_series = [all_features[i]["avg_future_ret"] for i in valid_indices]
ret_consistency_series = [all_features[i]["ret_sign_consistency"] for i in valid_indices]
cluster_ratio_series = [all_features[i]["match_cluster_ratio"] for i in valid_indices]

print(f"Scanned {len(scan_points)} points, {len(valid_indices)} valid ({len(scan_points) - len(valid_indices)} insufficient data)")

In [ ]:
# Plot feature evolution
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)

# F1: Best match quality
axes[0].plot(valid_dates, top1_sim_series, linewidth=1.5, color="#2c3e50")
axes[0].fill_between(valid_dates, 0, top1_sim_series, alpha=0.15, color="#2c3e50")
axes[0].axhline(y=0.8, color="#27ae60", linestyle="--", alpha=0.5, label="Strong pattern threshold (0.8)")
axes[0].set_ylabel("Top-1 Similarity")
axes[0].set_title("F1: Best Match Quality Over Time")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# F6: Forward return signals
color_ret = "#e74c3c" if np.mean(avg_future_ret_series) < 0 else "#27ae60"
axes[1].bar(valid_dates, avg_future_ret_series, width=8, color=color_ret, alpha=0.6, label="Avg future ret")
axes[1].set_ylabel("Average Forward Return")
axes[1].set_title("F6: Average Forward Return from Historical Matches")
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# F9: Positive return ratio (proportion of matches that were bullish)
axes[2].plot(valid_dates, ret_consistency_series, linewidth=1.5, color="#8e44ad")
axes[2].fill_between(valid_dates, 0.5, ret_consistency_series,
                      where=np.array(ret_consistency_series) > 0.5, alpha=0.2, color="#27ae60")
axes[2].fill_between(valid_dates, ret_consistency_series, 0.5,
                      where=np.array(ret_consistency_series) <= 0.5, alpha=0.2, color="#e74c3c")
axes[2].axhline(y=0.5, color="black", linestyle="--", alpha=0.5, label="50% (random)")
axes[2].set_ylabel("Positive Return Ratio")
axes[2].set_title("F9: Proportion of Matches with Positive Forward Returns")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1)

# F14: Cluster ratio
axes[3].plot(valid_dates, cluster_ratio_series, linewidth=1.5, color="#e67e22")
axes[3].axhline(y=0.5, color="#c0392b", linestyle="--", alpha=0.5, label="High clustering (>0.5 = concern)")
axes[3].set_ylabel("Cluster Ratio")
axes[3].set_title("F14: Match Clustering — Are matches concentrated in one period?")
axes[3].set_xlabel("Date")
axes[3].legend(fontsize=9)
axes[3].grid(True, alpha=0.3)
axes[3].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('05_feature_evolution.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('05_feature_evolution.png'))

## 4. Performance Benchmark: Rust/PyO3

The next cell times the installed Rust backend locally and visualizes the preregistered NRR-2026-023 wrapper-internal ratios. It does not import the C++ repository or a Python reference implementation.


In [ ]:
print("=" * 68)
print("PERFORMANCE: LOCAL RUST/PyO3 RUN + NRR-2026-023 SUMMARY")
print("=" * 68)

query = standardize_returns(prices[480:500])
candidate = standardize_returns(prices[300:320])
indices = np.arange(200, 750, 10, dtype=np.int64)

# Warm-up
for _ in range(20):
    dtw_distance(query, candidate, 5)
for _ in range(3):
    pattern_match_single(prices, 500)
pattern_match_batch(prices, indices)

n_dtw = 5000
t0 = time.perf_counter()
for _ in range(n_dtw):
    dtw_distance(query, candidate, 5)
local_dtw_us = (time.perf_counter() - t0) / n_dtw * 1e6

n_single = 50
t0 = time.perf_counter()
for _ in range(n_single):
    pattern_match_single(prices, 500)
local_single_ms = (time.perf_counter() - t0) / n_single * 1e3

t0 = time.perf_counter()
batch_features, batch_valid = pattern_match_batch(prices, indices)
local_batch_ms = (time.perf_counter() - t0) * 1e3

print(f"Local DTW L=19:        {local_dtw_us:.3f} µs/call")
print(f"Local pattern single:  {local_single_ms:.3f} ms/call")
print(f"Local batch ({len(indices)}): {local_batch_ms:.3f} ms/call")
print(f"Valid batch rows:      {int(batch_valid.sum())}/{len(batch_valid)}")

# Frozen wrapper-internal Rust/C++ median ratios from NRR-2026-023.
# Lower than 1.0 means Rust was faster; the 0.90–1.10 band is preregistered as tied.
labels = ["DTW L19\n1 thread", "Pattern single\n1 thread", "Batch 100\n1 thread", "Batch 100\n16 threads"]
ratios = np.array([0.318, 0.528, 0.920, 0.171])
colors = ["#27ae60" if value < 0.90 else "#f39c12" for value in ratios]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(["DTW (µs)", "Pattern (ms)", "Batch (ms)"],
            [local_dtw_us, local_single_ms, local_batch_ms],
            color=["#3498db", "#8e44ad", "#16a085"], alpha=0.8)
axes[0].set_title("Local installed Rust/PyO3 timing\n(different units; read labels only)")
axes[0].grid(True, alpha=0.3, axis="y")

bars = axes[1].bar(labels, ratios, color=colors, alpha=0.85)
axes[1].axhspan(0.90, 1.10, color="#f1c40f", alpha=0.16, label="preregistered tie band")
axes[1].axhline(1.0, color="black", linewidth=1)
axes[1].set_ylabel("Rust median / C++ median")
axes[1].set_title("NRR-2026-023 wrapper-internal medians")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, ratios):
    axes[1].text(bar.get_x() + bar.get_width()/2, value, f"{value:.3f}",
                 ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("08_benchmark.png", dpi=100, bbox_inches="tight", facecolor="white")
plt.close()
display(Image("08_benchmark.png"))

print("\nNRR note: most core metrics had CoV >5%, so findings are directional,")
print("not universal speed guarantees. Rust batch achieved ~5.33× self-speedup at 16 threads.")


## 5. Practical Use Case: Signal Generation

Combine pattern match features into a simple trading signal. This demonstrates how the 15 features feed into a strategy.

In [ ]:
def generate_pattern_signal(
    features: Dict[str, float],
    min_quality_matches: int = 2,
    consensus_threshold: float = 0.6,
    scale_factor: float = 20.0,      # scales daily-like return to signal magnitude
    cluster_floor: float = 0.05,       # minimum signal floor when cluster_ratio=1
    verbose: bool = False,
) -> float:
    """
    Convert 15-dimensional pattern features into a trading signal ∈ [-1, 1].

    Positive = bullish pattern (go long)
    Negative = bearish pattern (go short / exit)
    Near zero = no clear signal

    Gate structure (three explicit layers):
      Gate 1 (match quality):  n_matches_above_thresh >= min_quality_matches
      Gate 2 (outcome consensus): max(positive_ratio, 1-positive_ratio) >= consensus_threshold
      Gate 3 (position sizing):  magnitude × cluster_penalty → bounded signal

    This is a simplified example — production strategies would calibrate weights
    and run walk-forward cross-validation on real data.
    """
    if verbose:
        print("--- Gate checks ---")

    # ── Gate 1: Match quality ──
    # n_matches_above_thresh >= 2 already implies top1_sim > 0.8 (scores > 0.8
    # are counted, so >= 2 matches above 0.8 guarantees strong pattern quality)
    n_quality = features["n_matches_above_thresh"]
    if n_quality < min_quality_matches:
        if verbose:
            print(f"  G1 FAIL: n_matches_above_thresh={n_quality} < {min_quality_matches}")
        return 0.0
    if verbose:
        print(f"  G1 PASS: {n_quality} quality matches")

    # ── Gate 2: Outcome consensus (bidirectional) ──
    # Use true consistency: max(positive_ratio, 1-positive_ratio) ∈ [0.5, 1]
    # 0.5 = random, 1.0 = all matches agree on direction
    positive_ratio = features["ret_sign_consistency"]
    true_consistency = max(positive_ratio, 1.0 - positive_ratio)
    majority_direction = 1 if positive_ratio >= 0.5 else -1

    if true_consistency < consensus_threshold:
        if verbose:
            print(f"  G2 FAIL: consistency={true_consistency:.2f} < {consensus_threshold}")
        return 0.0
    if verbose:
        print(f"  G2 PASS: consistency={true_consistency:.2f} direction={'bull' if majority_direction > 0 else 'bear'}")

    # ── Gate 3: Position sizing ──
    weighted_ret = features["weighted_future_ret"]

    # Magnitude: scale the daily-like return to [-1, 1] range
    # scale_factor maps expected daily return (~0.05%) to a reasonable signal
    magnitude = min(abs(weighted_ret) * scale_factor, 1.0)

    # Cluster penalty: penalize matches concentrated in narrow time windows
    # (suggests overfitting to one specific market regime)
    # floor ensures signal doesn't vanish entirely when clustering is high
    cluster_ratio = features["match_cluster_ratio"]
    cluster_penalty = max(1.0 - cluster_ratio, cluster_floor)

    signal = majority_direction * magnitude * cluster_penalty

    if verbose:
        print(f"  G3: magnitude={magnitude:.3f} cluster_penalty={cluster_penalty:.3f} → signal={signal:.3f}")

    return float(np.clip(signal, -1.0, 1.0))


# Scan for signals across time
scan_T = list(range(200, 780, 5))
signals = []
signal_dates = []

for T in scan_T:
    feats = pattern_match_single(prices, T)
    if feats is not None:
        sig = generate_pattern_signal(feats)
        signals.append(sig)
        signal_dates.append(dates[T])
    else:
        signals.append(0.0)
        signal_dates.append(dates[T])

signals = np.array(signals)
signal_dates_arr = np.array(signal_dates)

# Count signal events
bullish = np.sum(signals > 0.1)
bearish = np.sum(signals < -0.1)
neutral = np.sum((signals >= -0.1) & (signals <= 0.1))

# Show one example with verbose gate diagnostics
print("Example gate trace (T_idx=500):")
_example_feats = pattern_match_single(prices, 500)
if _example_feats:
    _ = generate_pattern_signal(_example_feats, verbose=True)

print(f"\nSignal scan: {len(scan_T)} points")
print(f"  Bullish (>0.1):  {bullish:3d} ({bullish/len(scan_T):.0%})")
print(f"  Bearish (<-0.1): {bearish:3d} ({bearish/len(scan_T):.0%})")
print(f"  Neutral:         {neutral:3d} ({neutral/len(scan_T):.0%})")

In [ ]:
# ⚠️ TOY EXAMPLE — NOT INVESTMENT ADVICE ⚠️
# This simplified walk-forward illustration uses synthetic data and
# next-day execution assumptions. It does NOT reflect real trading
# (no transaction costs, no slippage, no same-bar constraints).
# Do NOT use this as a basis for actual investment decisions.
#
# Visualize signals on price chart
fig, axes = plt.subplots(3, 1, figsize=(15, 10), height_ratios=[3, 1, 1], sharex=True)

# Price
axes[0].plot(dates, prices, linewidth=0.8, color="#1a3a5c")
axes[0].set_ylabel("Price ($)")
axes[0].set_title("Pattern-Matching Trading Signals (Toy Example — Not Investment Advice)")
axes[0].grid(True, alpha=0.3)

# Mark signal points on price
bull_mask = signals > 0.1
bear_mask = signals < -0.1
if np.any(bull_mask):
    axes[0].scatter(signal_dates_arr[bull_mask], prices[scan_T][bull_mask],
                    color="#27ae60", s=40, marker="^", alpha=0.8, zorder=5, label="Bullish")
if np.any(bear_mask):
    axes[0].scatter(signal_dates_arr[bear_mask], prices[scan_T][bear_mask],
                    color="#e74c3c", s=40, marker="v", alpha=0.8, zorder=5, label="Bearish")
axes[0].legend(fontsize=9)

# Signal strength
axes[1].bar(signal_dates_arr, signals, width=5,
            color=["#27ae60" if s > 0 else "#e74c3c" if s < 0 else "#bdc3c7" for s in signals], alpha=0.7)
axes[1].axhline(y=0.1, color="#27ae60", linestyle="--", alpha=0.3, linewidth=0.5)
axes[1].axhline(y=-0.1, color="#e74c3c", linestyle="--", alpha=0.3, linewidth=0.5)
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].set_ylabel("Signal")
axes[1].set_ylim(-1.1, 1.1)
axes[1].grid(True, alpha=0.3)

# Cumulative return from following signals
# (toy illustration: next-day execution, signal strength as position weight,
#  5-day holding period, geometrically compounded)
cumulative_pnl = [1.0]  # start at 1.0 for geometric compounding
for i, T in enumerate(scan_T):
    sig = signals[i]
    if T + 5 < len(prices) and abs(sig) > 0.1:
        # Next-day execution to avoid same-bar bias
        entry_price = prices[T + 1]
        exit_price = prices[min(T + 1 + 5, len(prices) - 1)]
        fut_ret = exit_price / entry_price - 1
        # Signal strength determines position size and direction
        pnl_contrib = sig * fut_ret
    else:
        pnl_contrib = 0.0
    cumulative_pnl.append(cumulative_pnl[-1] * (1.0 + pnl_contrib))
cumulative_pnl = np.array(cumulative_pnl[1:])

# Display as percentage return
cumulative_pct = (cumulative_pnl - 1.0) * 100
axes[2].plot(signal_dates_arr, cumulative_pct, linewidth=1.5, color="#8e44ad")
axes[2].fill_between(signal_dates_arr, 0, cumulative_pct,
                      where=cumulative_pct > 0, alpha=0.15, color="#27ae60")
axes[2].fill_between(signal_dates_arr, 0, cumulative_pct,
                      where=cumulative_pct <= 0, alpha=0.15, color="#e74c3c")
axes[2].axhline(y=0, color="black", linewidth=0.5)
axes[2].set_ylabel("Cumulative Return (%)")
axes[2].set_xlabel("Date")
axes[2].grid(True, alpha=0.3)
# Add disclaimer watermark
axes[2].text(0.99, 0.02, "TOY EXAMPLE — not real trading performance",
             transform=axes[2].transAxes, fontsize=8, color="gray",
             ha="right", va="bottom", style="italic")

plt.tight_layout()
plt.savefig('06_trading_signals.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('06_trading_signals.png'))

## 6. Batch DTW: Compare One Pattern Against Many

The `dtw_distance_batch` function computes DTW distances between one query and many candidates efficiently. This is useful for:

- Finding the most similar ETFs to a target
- Screening a universe of symbols for pattern similarity
- Pre-computing distance matrices for clustering

In [ ]:
# Simulate: compare one query ETF to a universe of 50 candidate ETFs
n_etfs = 50
window_len = 19  # L_query - 1
rng = np.random.default_rng(123)

query_etf = standardize_returns(prices[480:500])
candidate_etfs = np.empty((n_etfs, window_len))

for i in range(n_etfs):
    noise_level = 0.3 + 0.7 * (i / n_etfs)
    candidate_etfs[i] = (1 - noise_level) * query_etf + noise_level * rng.standard_normal(window_len)

# The helper calls the Rust `dtw_distance` function for every candidate.
t0 = time.perf_counter()
top_idx, top_dists = dtw_distance_batch(query_etf, candidate_etfs, window=5, top_k=10)
elapsed = (time.perf_counter() - t0) * 1e3
engine = "Rust/PyO3"

print(f"Batch DTW: 1 query × {n_etfs} candidates [{engine}]")
print(f"  Time: {elapsed:.2f} ms")
print("\n  Top-10 most similar ETFs:")
for rank, (idx, dist) in enumerate(zip(top_idx, top_dists)):
    marker = " ★" if rank == 0 else ""
    print(f"    {rank+1:2d}. ETF #{idx:2d}  dtw_distance={dist:.6f}{marker}")


In [ ]:
# Visualize: Query ETF vs Top-3 most similar ETFs
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_top3 = ["#e74c3c", "#2ecc71", "#3498db"]

# Left: DTW distance distribution
all_dists = np.array([dtw_distance(query_etf, c, window=5) for c in candidate_etfs])
axes[0].hist(all_dists, bins=20, color="#7f8c8d", edgecolor="white", alpha=0.7)
for rank, (idx, dist) in enumerate(zip(top_idx[:3], top_dists[:3])):
    axes[0].axvline(x=dist, color=colors_top3[rank], linewidth=2, linestyle="--",
                    label=f"#{rank+1}: ETF #{idx} ({dist:.4f})")
axes[0].set_title(f"DTW Distance Distribution ({n_etfs} ETFs)")
axes[0].set_xlabel("DTW Distance (lower = more similar)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Right: Overlay query vs top-3
x_axis = np.arange(window_len)
axes[1].plot(x_axis, query_etf, "o-", linewidth=3, markersize=6, color="black", label="Query ETF", zorder=10)
for rank, (idx, dist) in enumerate(zip(top_idx[:3], top_dists[:3])):
    axes[1].plot(x_axis, candidate_etfs[idx], "s--", linewidth=1.5, markersize=4,
                 color=colors_top3[rank], alpha=0.7, label=f"#{rank+1}: ETF #{idx} (d={dist:.4f})")
axes[1].set_title("Query ETF vs Top-3 Most Similar ETFs (Standardized Returns)")
axes[1].set_xlabel("Day offset")
axes[1].set_ylabel("Standardized Return")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('07_batch_dtw.png', dpi=100, bbox_inches='tight', facecolor='white')
plt.close()
display(Image('07_batch_dtw.png'))

## 7. API Reference Summary

| Public symbol | Description | Rust/PyO3 backend |
|:---|:---|:---:|
| `standardize_returns(prices)` | Prices → standardized log returns | ✅ |
| `cosine_similarity(x, y)` | Cosine similarity in `[-1, 1]` | ✅ |
| `dtw_distance(x, y, window=5)` | Sakoe-Chiba constrained DTW | ✅ |
| `pattern_match_single(prices, T_idx, ...)` | Full pipeline → 15 features or `None` | ✅ |
| `pattern_match_batch(prices, t_indices, ...)` | rayon batch → `(features, valid_mask)` | ✅ |
| `FEATURE_KEYS` | Stable ordering of the 15 feature names | ✅ |

The notebook-only `generate_query_candidates` and `dtw_distance_batch` helpers are educational orchestration helpers. Their numerical kernels call the installed Rust package; they are not additional package APIs.

---

## Next Steps

- Replace synthetic data with a governed, reproducible ETF dataset.
- Calibrate signal weights with walk-forward validation.
- Treat the signal section as a toy illustration, not investment advice.
- Re-run benchmarks on the target hardware instead of treating NRR results as universal.
- Use the optional Panel adapter when integrating with `ml-quant-trading`.

---

> **Project**: [github.com/redamancy231-create/etf-pattern-match-pyo3](https://github.com/redamancy231-create/etf-pattern-match-pyo3)  
> **License**: MIT  
> **Built with**: Python 3.12 · NumPy · Rust 1.97+ · PyO3 · rayon
